# Generating Data

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import time

In [1]:
# ------------------ Hyperparameters ------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# hyperparameters
batch_size = 8 # how many independent sequences will we process in parallel?
block_size = 20 # what is the maximum context length for predictions? Sequence length
max_iters = 3500
eval_interval = 100
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 128 # embedding dimension
n_head = 8 # number of heads
n_layer = 6
dropout = 0.2

NameError: name 'torch' is not defined

In [ ]:
words = []

# 1 to 19
ones = [
    "one","two","three","four","five",
    "six","seven","eight","nine","ten",
    "eleven","twelve","thirteen","fourteen","fifteen",
    "sixteen","seventeen","eighteen","nineteen"
]

# tens words (20, 30, ..., 90)
tens = ["twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]

# add 1–19
words.extend(ones)

# add 20–99
for t in tens:
    words.append(t)  # exact 20, 30, ...
    for o in ["one","two","three","four","five","six","seven","eight","nine"]:
        words.append(t)
        words.append(o)

# add 100–500
for h in ["one","two","three","four","five"]:
    words.append(h)
    words.append("hundred")
    # 101–119, etc.
    for o in ones :
      words.append(h)
      words.append("hundred")
      words.append(o)

    # words.extend([h, "hundred", o] for o in ones)
    # 120–199, etc.
    for t in tens:
        words.append(h)
        words.append("hundred")
        words.append(t)
        for o in ["one","two","three","four","five","six","seven","eight","nine"]:
            words.append(h)
            words.append("hundred")
            words.append(t)
            words.append(o)

print(words[100:300])   # preview first 200 tokens
print(len(words))    # total vocabulary size


['sixty', 'three', 'sixty', 'four', 'sixty', 'five', 'sixty', 'six', 'sixty', 'seven', 'sixty', 'eight', 'sixty', 'nine', 'seventy', 'seventy', 'one', 'seventy', 'two', 'seventy', 'three', 'seventy', 'four', 'seventy', 'five', 'seventy', 'six', 'seventy', 'seven', 'seventy', 'eight', 'seventy', 'nine', 'eighty', 'eighty', 'one', 'eighty', 'two', 'eighty', 'three', 'eighty', 'four', 'eighty', 'five', 'eighty', 'six', 'eighty', 'seven', 'eighty', 'eight', 'eighty', 'nine', 'ninety', 'ninety', 'one', 'ninety', 'two', 'ninety', 'three', 'ninety', 'four', 'ninety', 'five', 'ninety', 'six', 'ninety', 'seven', 'ninety', 'eight', 'ninety', 'nine', 'one', 'hundred', 'one', 'hundred', 'one', 'one', 'hundred', 'two', 'one', 'hundred', 'three', 'one', 'hundred', 'four', 'one', 'hundred', 'five', 'one', 'hundred', 'six', 'one', 'hundred', 'seven', 'one', 'hundred', 'eight', 'one', 'hundred', 'nine', 'one', 'hundred', 'ten', 'one', 'hundred', 'eleven', 'one', 'hundred', 'twelve', 'one', 'hundred', '

In [ ]:
set(words)

{'eight',
 'eighteen',
 'eighty',
 'eleven',
 'fifteen',
 'fifty',
 'five',
 'forty',
 'four',
 'fourteen',
 'hundred',
 'nine',
 'nineteen',
 'ninety',
 'one',
 'seven',
 'seventeen',
 'seventy',
 'six',
 'sixteen',
 'sixty',
 'ten',
 'thirteen',
 'thirty',
 'three',
 'twelve',
 'twenty',
 'two'}

In [ ]:
# Vocabulary:
vocab = set(words)

# Make word -> index and index -> word mappings
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for i, w in enumerate(vocab)}

vocab_size = len(vocab)
print(vocab_size)

28


In [ ]:
for _ in range(10):
  words.extend(words)

In [ ]:
len(words)

2074624

In [ ]:
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ' '.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

In [ ]:
data = torch.tensor(encode(words), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]


In [ ]:
len(val_data)

207463

In [ ]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
x,y = get_batch('train')

In [ ]:
x,y

(tensor([[ 6, 24, 18, 14, 12, 24, 18, 14, 24, 24, 18, 14,  4, 24, 18, 14,  5, 24,
          18, 14],
         [15, 18,  0,  5, 15, 18,  0,  2, 15, 18,  0, 13, 15, 18,  8, 15, 18,  8,
           3, 15],
         [24,  3, 18, 20,  4,  3, 18, 20,  5,  3, 18, 20,  2,  3, 18, 20, 13,  3,
          18,  0],
         [15, 18,  0,  2, 15, 18,  0, 13, 15, 18,  8, 15, 18,  8,  3, 15, 18,  8,
          15, 15],
         [19, 13, 20, 20,  3, 20, 15, 20,  6, 20, 12, 20, 24, 20,  4, 20,  5, 20,
           2, 20],
         [24,  6, 18, 14,  4,  6, 18, 14,  5,  6, 18, 14,  2,  6, 18, 14, 13,  6,
          18, 25],
         [ 2,  9, 13, 19, 19,  3, 19, 15, 19,  6, 19, 12, 19, 24, 19,  4, 19,  5,
          19,  2],
         [ 0,  6, 24, 18,  0, 12, 24, 18,  0, 24, 24, 18,  0,  4, 24, 18,  0,  5,
          24, 18]]),
 tensor([[24, 18, 14, 12, 24, 18, 14, 24, 24, 18, 14,  4, 24, 18, 14,  5, 24, 18,
          14,  2],
         [18,  0,  5, 15, 18,  0,  2, 15, 18,  0, 13, 15, 18,  8, 15, 18,  8,  3,
       

In [ ]:
n = 2
for i in range(len(x[n].tolist())):
  context = decode(x[n].tolist()[0:i+1])
  target = itos[y[n].tolist()[i]]
  print(f"When input is {context}, --> target is {target}")

When input is five, --> target is one
When input is five one, --> target is hundred
When input is five one hundred, --> target is forty
When input is five one hundred forty, --> target is six
When input is five one hundred forty six, --> target is one
When input is five one hundred forty six one, --> target is hundred
When input is five one hundred forty six one hundred, --> target is forty
When input is five one hundred forty six one hundred forty, --> target is seven
When input is five one hundred forty six one hundred forty seven, --> target is one
When input is five one hundred forty six one hundred forty seven one, --> target is hundred
When input is five one hundred forty six one hundred forty seven one hundred, --> target is forty
When input is five one hundred forty six one hundred forty seven one hundred forty, --> target is eight
When input is five one hundred forty six one hundred forty seven one hundred forty eight, --> target is one
When input is five one hundred forty six

#Training the Model

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

In [ ]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [ ]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [ ]:
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

1.19734 M parameters


In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


In [ ]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    # if iter > 50 :
    #   break

step 0: train loss 3.4608, val loss 3.4610
step 100: train loss 1.5844, val loss 1.5979
step 200: train loss 1.1204, val loss 1.1270
step 300: train loss 0.8054, val loss 0.8001
step 400: train loss 0.6255, val loss 0.6218
step 500: train loss 0.5161, val loss 0.5102
step 600: train loss 0.4110, val loss 0.4083
step 700: train loss 0.3830, val loss 0.3774
step 800: train loss 0.3485, val loss 0.3371
step 900: train loss 0.3218, val loss 0.3242
step 1000: train loss 0.3113, val loss 0.3215
step 1100: train loss 0.3056, val loss 0.3087
step 1200: train loss 0.3052, val loss 0.3095
step 1300: train loss 0.3039, val loss 0.2981
step 1400: train loss 0.2886, val loss 0.2877
step 1500: train loss 0.2988, val loss 0.3001
step 1600: train loss 0.2891, val loss 0.2903
step 1700: train loss 0.2793, val loss 0.2795
step 1800: train loss 0.2812, val loss 0.2822
step 1900: train loss 0.2751, val loss 0.2737
step 2000: train loss 0.2756, val loss 0.2789
step 2100: train loss 0.2760, val loss 0.2762


In [ ]:
# prompt = torch.tensor([[stoi["thirty"], stoi["one"]]], dtype=torch.long, device=device)
# output = m.generate(prompt, max_new_tokens=20)
# print(decode(output[0].tolist()))

In [ ]:
def generate(prompt_input):
  prompt_input_list = prompt_input.split()
  prompt_input_list = prompt_input_list[-block_size:]
  prompt_input_stoi_ids = encode(prompt_input_list)
  prompt_final = torch.tensor([prompt_input_stoi_ids], dtype=torch.long, device=device)
  output_ids = m.generate(prompt_final, max_new_tokens=100)
  return decode(output_ids[0].tolist())

In [ ]:
prompt_input = "four hundred twenty one four hundred twenty two four hundred twenty three"
generate(prompt_input)

'four hundred twenty one four hundred twenty two four hundred twenty three four hundred twenty four four hundred twenty five four hundred twenty six four hundred twenty seven four hundred twenty eight four hundred twenty nine four hundred thirty four hundred thirty one four hundred thirty two four hundred thirty three four hundred thirty four four hundred thirty five four hundred thirty six four hundred thirty seven four hundred thirty eight four hundred thirty nine four hundred forty four hundred forty one four hundred forty two four hundred forty three four hundred forty three four hundred forty four four hundred forty five four hundred forty six four hundred forty seven four hundred'

#Save Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
save_path = "/content/drive/MyDrive/gpt_language_model.pth"
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")

Model saved to /content/drive/MyDrive/gpt_language_model.pth
